# YOLO11 Crack Segmentation — COCO → YOLO Pipeline

**What this notebook does:**
1. Validates GPU availability
2. Installs Ultralytics + Supervision (no Roboflow required)
3. Mounts Google Drive and points to your COCO dataset
4. Converts COCO segmentation JSON → YOLO `.txt` labels automatically
5. Trains `yolo11n-seg.pt` (fast sanity check) and `yolo11s-seg.pt` (full accuracy run)
6. Validates and runs inference on your crack images with annotated visualisations

**Expected COCO folder structure on Drive:**
```
MyDrive/crack_dataset/
├── train/
│   ├── images/
│   └── _annotations.coco.json
├── valid/
│   ├── images/
│   └── _annotations.coco.json
└── test/
    ├── images/
    └── _annotations.coco.json
```
If your JSON file has a different name just update `COCO_JSON_NAME` below.

## 1 · Check GPU

In [ ]:
!nvidia-smi

## 2 · Install Dependencies
No Roboflow account needed.

In [ ]:
%pip install -q ultralytics supervision
!yolo settings sync=False
import ultralytics
ultralytics.checks()

## 3 · Mount Google Drive
Upload your COCO dataset to Drive first, then adjust `DRIVE_DATASET_PATH` below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
HOME = '/content'

# ── USER CONFIG ──────────────────────────────────────────────────────────────
DRIVE_DATASET_PATH = '/content/drive/MyDrive/crack_dataset'  # <-- edit this
COCO_JSON_NAME     = '_annotations.coco.json'                # <-- JSON filename in each split
SPLITS             = ['train', 'valid', 'test']              # folder names that exist
# ─────────────────────────────────────────────────────────────────────────────

YOLO_DATASET_PATH = os.path.join(HOME, 'crack_yolo')
print('Source:', DRIVE_DATASET_PATH)
print('YOLO dataset path:', YOLO_DATASET_PATH)

## 4 · Convert COCO Segmentation → YOLO Format

YOLO segmentation label format per line:
```
<class_id>  <x1> <y1>  <x2> <y2>  ...  <xn> <yn>   (all normalised 0-1)
```
The converter handles multiple polygons per annotation, skips crowd masks, and symlinks images so no data is duplicated.

In [ ]:
import json, os, shutil
from pathlib import Path
from collections import defaultdict


def coco_to_yolo_seg(coco_json_path, images_dir, out_labels_dir, out_images_dir,
                     cat_id_to_yolo=None):
    """Convert one COCO split to YOLO segmentation labels.
    Returns the cat_id_to_yolo mapping so it can be reused across splits.
    """
    with open(coco_json_path) as f:
        coco = json.load(f)

    if cat_id_to_yolo is None:
        cats = sorted(coco['categories'], key=lambda c: c['id'])
        cat_id_to_yolo = {c['id']: i for i, c in enumerate(cats)}

    img_meta = {img['id']: img for img in coco['images']}

    ann_by_img = defaultdict(list)
    for ann in coco['annotations']:
        if ann.get('iscrowd', 0):
            continue
        ann_by_img[ann['image_id']].append(ann)

    out_labels_dir = Path(out_labels_dir)
    out_images_dir = Path(out_images_dir)
    out_labels_dir.mkdir(parents=True, exist_ok=True)
    out_images_dir.mkdir(parents=True, exist_ok=True)

    skipped = 0
    for img_id, img_info in img_meta.items():
        W, H  = img_info['width'], img_info['height']
        fname = img_info['file_name']
        stem  = Path(fname).stem

        # Symlink (or copy) image
        src_img = Path(images_dir) / fname
        dst_img = out_images_dir / fname
        if src_img.exists() and not dst_img.exists():
            try:
                os.symlink(src_img.resolve(), dst_img)
            except Exception:
                shutil.copy2(src_img, dst_img)

        anns = ann_by_img.get(img_id, [])
        if not anns:
            continue

        label_lines = []
        for ann in anns:
            cls  = cat_id_to_yolo.get(ann['category_id'])
            segs = ann.get('segmentation', [])
            if cls is None or not segs or not isinstance(segs, list):
                skipped += 1
                continue
            # Take the polygon with the most points when multiple exist
            poly = max(segs, key=len) if isinstance(segs[0], list) else segs
            if len(poly) < 6:
                skipped += 1
                continue
            coords = poly if isinstance(poly[0], (int, float)) else poly[0]
            norm   = []
            for i in range(0, len(coords) - 1, 2):
                nx = max(0.0, min(1.0, coords[i]   / W))
                ny = max(0.0, min(1.0, coords[i+1] / H))
                norm.extend([f'{nx:.6f}', f'{ny:.6f}'])
            label_lines.append(f"{cls} {' '.join(norm)}")

        if label_lines:
            (out_labels_dir / f'{stem}.txt').write_text('\n'.join(label_lines))

    print(f'  {len(img_meta)} images | {skipped} annotations skipped')
    return cat_id_to_yolo


# Run conversion for every available split
cat_map = None
for split in SPLITS:
    split_dir  = Path(DRIVE_DATASET_PATH) / split
    coco_json  = split_dir / COCO_JSON_NAME
    images_src = split_dir / 'images'
    if not coco_json.exists():
        print(f'[SKIP] {split}: {coco_json} not found')
        continue
    print(f'Converting {split}...')
    cat_map = coco_to_yolo_seg(
        coco_json_path = coco_json,
        images_dir     = images_src,
        out_labels_dir = Path(YOLO_DATASET_PATH) / split / 'labels',
        out_images_dir = Path(YOLO_DATASET_PATH) / split / 'images',
        cat_id_to_yolo = cat_map,
    )

print('\nCategory map (COCO id → YOLO class index):', cat_map)

## 5 · Generate `data.yaml`

In [ ]:
import yaml

def get_class_names(dataset_path, splits, json_name):
    for split in splits:
        p = Path(dataset_path) / split / json_name
        if p.exists():
            with open(p) as f:
                coco = json.load(f)
            return [c['name'] for c in sorted(coco['categories'], key=lambda c: c['id'])]
    raise FileNotFoundError('No COCO JSON found')

class_names = get_class_names(DRIVE_DATASET_PATH, SPLITS, COCO_JSON_NAME)
print('Classes:', class_names)

split_paths = {}
for split in SPLITS:
    p = Path(YOLO_DATASET_PATH) / split / 'images'
    if p.exists() and any(p.iterdir()):
        split_paths[split] = str(p)

yaml_data = dict(
    path  = YOLO_DATASET_PATH,
    train = split_paths.get('train', 'train/images'),
    val   = split_paths.get('valid', 'valid/images'),
    test  = split_paths.get('test',  'test/images'),
    nc    = len(class_names),
    names = class_names,
)

YAML_PATH = os.path.join(YOLO_DATASET_PATH, 'data.yaml')
with open(YAML_PATH, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False)

print(open(YAML_PATH).read())

## 6 · Label Sanity Check
Visualises converted masks on 3 random training images before wasting GPU time.

In [ ]:
import random, numpy as np
from pathlib import Path
from PIL import Image as PILImage, ImageDraw
import supervision as sv
from IPython.display import display

train_img_dir   = Path(YOLO_DATASET_PATH) / 'train' / 'images'
train_label_dir = Path(YOLO_DATASET_PATH) / 'train' / 'labels'
imgs = list(train_img_dir.glob('*.[jp][pn]g'))
sample = random.sample(imgs, min(3, len(imgs)))

for img_path in sample:
    label_path = train_label_dir / (img_path.stem + '.txt')
    if not label_path.exists():
        print(f'No label: {img_path.name}'); continue

    pil = PILImage.open(img_path).convert('RGB')
    W, H = pil.size
    arr  = np.array(pil)

    masks, cls_ids = [], []
    for line in label_path.read_text().strip().splitlines():
        parts  = list(map(float, line.split()))
        cls_ids.append(int(parts[0]))
        coords = parts[1:]
        xs = [int(coords[i]   * W) for i in range(0, len(coords), 2)]
        ys = [int(coords[i+1] * H) for i in range(0, len(coords), 2)]
        m  = PILImage.new('L', (W, H), 0)
        ImageDraw.Draw(m).polygon(list(zip(xs, ys)), fill=1)
        masks.append(np.array(m).astype(bool))

    if not masks:
        print(f'Empty: {label_path.name}'); continue

    dets = sv.Detections(
        xyxy     = sv.mask_to_xyxy(np.stack(masks)),
        mask     = np.stack(masks),
        class_id = np.array(cls_ids),
    )
    ann = sv.MaskAnnotator(opacity=0.45).annotate(arr.copy(), dets)
    ann = sv.LabelAnnotator(text_color=sv.Color.WHITE).annotate(
        ann, dets, labels=[class_names[c] for c in cls_ids])
    sv.plot_image(ann, size=(8, 8))
    print(img_path.name, '—', len(masks), 'annotations')

## 7 · Train — `yolo11n-seg.pt` (Fast Sanity Run, ~5 min)

> Skip to **Section 8** if your sanity check looks correct.

In [ ]:
import os
%cd {HOME}

!yolo segment train \
    model=yolo11n-seg.pt \
    data={YAML_PATH} \
    epochs=20 \
    imgsz=640 \
    batch=16 \
    patience=10 \
    plots=True \
    project={HOME}/runs/segment \
    name=crack_nano \
    exist_ok=True

In [ ]:
from IPython.display import Image as IPyImage
IPyImage(filename=f'{HOME}/runs/segment/crack_nano/results.png', width=900)

## 8 · Train — `yolo11s-seg.pt` (Full Accuracy Run)

**Crack-optimised hyperparameters:**

| Parameter | Value | Reason |
|---|---|---|
| `imgsz` | 1280 | Cracks are thin — higher resolution captures fine detail |
| `batch` | 8 | Memory-safe at 1280 px on T4 |
| `epochs` | 150 | Crack textures need more exposure to generalise |
| `patience` | 30 | Early-stop if no mAP gain for 30 epochs |
| `close_mosaic` | 15 | Disable mosaic in final 15 epochs for cleaner convergence |
| `hsv_s` | 0.3 | Mild saturation shift — cracks are mostly greyscale |
| `degrees` | 15 | Small rotation — cracks appear at any angle |
| `fliplr/ud` | 0.5 | Both flips valid for crack orientation |
| `overlap_mask` | True | Required when multiple crack instances overlap |
| `mask_ratio` | 1 | Full-resolution masks — no downsampling |

> **A100/V100:** bump `batch=16` and `imgsz=1536` for even better results.

In [ ]:
%cd {HOME}

!yolo segment train \
    model=yolo11s-seg.pt \
    data={YAML_PATH} \
    epochs=150 \
    imgsz=1280 \
    batch=8 \
    patience=30 \
    mosaic=1.0 \
    close_mosaic=15 \
    hsv_s=0.3 \
    degrees=15.0 \
    fliplr=0.5 \
    flipud=0.5 \
    overlap_mask=True \
    mask_ratio=1 \
    plots=True \
    project={HOME}/runs/segment \
    name=crack_small \
    exist_ok=True

BEST_MODEL = f'{HOME}/runs/segment/crack_small/weights/best.pt'
print('Best weights:', BEST_MODEL)

## 9 · Review Training Results

In [ ]:
from IPython.display import Image as IPyImage, display
import os

for fname in ['results.png', 'confusion_matrix_normalized.png',
              'val_batch0_labels.jpg', 'val_batch0_pred.jpg']:
    path = f'{HOME}/runs/segment/crack_small/{fname}'
    if os.path.exists(path):
        print(f'── {fname} ─────────────')
        display(IPyImage(filename=path, width=900))

## 10 · Validate on Test Set
Reports `mAP50`, `mAP50-95`, Precision, Recall and per-class mask metrics.

In [ ]:
!yolo segment val \
    model={BEST_MODEL} \
    data={YAML_PATH} \
    imgsz=1280 \
    split=test \
    plots=True \
    project={HOME}/runs/segment \
    name=crack_val \
    exist_ok=True

In [ ]:
from IPython.display import Image as IPyImage, display
import os

for fname in ['confusion_matrix_normalized.png', 'val_batch0_pred.jpg']:
    path = f'{HOME}/runs/segment/crack_val/{fname}'
    if os.path.exists(path):
        print(fname)
        display(IPyImage(filename=path, width=900))

## 11 · Predict on Test Images

> Lower `conf` (e.g. 0.15) if cracks are subtle. Raise `iou` (e.g. 0.7) if you see duplicate detections.

In [ ]:
from pathlib import Path

TEST_IMAGES_DIR = str(Path(YOLO_DATASET_PATH) / 'test' / 'images')

!yolo segment predict \
    model={BEST_MODEL} \
    source={TEST_IMAGES_DIR} \
    imgsz=1280 \
    conf=0.25 \
    iou=0.5 \
    save=True \
    save_txt=True \
    project={HOME}/runs/segment \
    name=crack_predict \
    exist_ok=True

## 12 · Visualise Predictions with Supervision

In [ ]:
import glob, random, os, numpy as np
import supervision as sv
from ultralytics import YOLO
from PIL import Image as PILImage
from IPython.display import display

model      = YOLO(BEST_MODEL)
test_imgs  = glob.glob(f'{TEST_IMAGES_DIR}/*.[jp][pn]g')
sample     = random.sample(test_imgs, min(6, len(test_imgs)))

mask_ann  = sv.MaskAnnotator(opacity=0.45, color_lookup=sv.ColorLookup.CLASS)
box_ann   = sv.BoxAnnotator(thickness=2)
label_ann = sv.LabelAnnotator(text_color=sv.Color.WHITE, text_scale=0.6)

for img_path in sample:
    result = model.predict(img_path, imgsz=1280, conf=0.25, iou=0.5, verbose=False)[0]
    dets   = sv.Detections.from_ultralytics(result)
    arr    = np.array(PILImage.open(img_path).convert('RGB'))

    ann = mask_ann.annotate(arr.copy(), dets)
    ann = box_ann.annotate(ann, dets)
    if len(dets):
        labels = [f"{class_names[int(c)]} {s:.2f}"
                  for c, s in zip(dets.class_id, dets.confidence)]
        ann = label_ann.annotate(ann, dets, labels=labels)

    print(f'{os.path.basename(img_path)}  →  {len(dets)} crack(s)')
    sv.plot_image(ann, size=(10, 10))

## 13 · Export to ONNX (Optional)

In [ ]:
from ultralytics import YOLO

model     = YOLO(BEST_MODEL)
onnx_path = model.export(format='onnx', imgsz=1280, simplify=True)
print('ONNX saved to:', onnx_path)

# TensorRT for Jetson / edge — uncomment if needed:
# trt_path = model.export(format='engine', imgsz=1280, half=True)

## 14 · Save Weights to Google Drive

In [ ]:
import shutil, datetime, os

ts        = datetime.datetime.now().strftime('%Y%m%d_%H%M')
drive_out = f'/content/drive/MyDrive/crack_yolo_weights_{ts}'
os.makedirs(drive_out, exist_ok=True)

shutil.copy(BEST_MODEL, f'{drive_out}/best.pt')
shutil.copy(YAML_PATH,  f'{drive_out}/data.yaml')

if 'onnx_path' in dir() and os.path.exists(str(onnx_path)):
    shutil.copy(str(onnx_path), f'{drive_out}/best.onnx')

print('Saved to Drive:', drive_out)
print(os.listdir(drive_out))